[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/example1_toggle_switch.ipynb)

# Example 1. Toggle switch

Two mutually repressing genes, the standard bistable two-gene circuit. The notebook samples
true parameters from the bistable DSGRN region, generates noisy synthetic trajectories, and
recovers the parameters two ways. Gradient-matching least squares fits `(L, U, theta)` with the
steepness `d` and decay `gamma` held fixed; the PINN learns all of `(L, U, theta, d, gamma)`
jointly with the network weights. Each recovery is scored by region membership (the explicit
DSGRN inequality, no DSGRN install) rather than parameter error.

In [ ]:
# Colab already has numpy/scipy/matplotlib/torch, so this cell is a no-op there.
# Uncomment if you are running somewhere that is missing a package.
# !pip -q install numpy scipy matplotlib torch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares
import torch, torch.nn as nn
torch.set_default_dtype(torch.float64)
np.set_printoptions(precision=3, suppress=True)

### Hill production functions

Production is built from Hill functions: smooth switches between a low level `L` and a high
level `U` across a threshold `theta`, with steepness `d`. `f+` is increasing (activation),
`f-` decreasing (repression), with `f+ + f- = L + U`. Defined twice below, once in NumPy for
simulation and once in torch for the PINN's physics loss.

In [ ]:
# --- Hill production functions: smooth switches (gamma normalized to 1) ---
def hill_act(x, L, U, th, d):   # activating: low L -> high U as x rises
    xd = np.power(x, d); td = np.power(th, d)
    return L + (U - L) * xd / (td + xd)

def hill_rep(x, L, U, th, d):   # repressing: high U -> low L as x rises
    xd = np.power(x, d); td = np.power(th, d)
    return L + (U - L) * td / (td + xd)

In [ ]:
# --- the same two functions in torch (used inside the network's physics loss) ---
def hill_act_t(x, L, U, th, d):
    xd = x.clamp(min=1e-9).pow(d); td = th.pow(d)
    return L + (U - L) * xd / (td + xd)

def hill_rep_t(x, L, U, th, d):
    xd = x.clamp(min=1e-9).pow(d); td = th.pow(d)
    return L + (U - L) * td / (td + xd)

## 1. Model and DSGRN region

$$\dot{x}_1=-x_1+f^-(x_2),\qquad \dot{x}_2=-x_2+f^-(x_1),\qquad \gamma=1.$$

The bistable region is
$$0 < L < \gamma\,\theta < U,$$
the threshold lying between the low and high production levels. `in_region` below is this
inequality, evaluated directly on the parameters.

In [ ]:
P = dict(L=1.0, U=5.0, th=3.0, d=10.0, g=1.0)   # 'true' parameters, chosen in the region

def rhs(t, x, p):
    x1, x2 = x
    return [-p['g']*x1 + hill_rep(x2, p['L'], p['U'], p['th'], p['d']),
            -p['g']*x2 + hill_rep(x1, p['L'], p['U'], p['th'], p['d'])]

def in_region(p):   # DSGRN toggle node:  0 < L < gamma*theta < U
    return (0 < p['L']) and (p['L'] < p['g']*p['th'] < p['U'])

print('do the true parameters lie in the bistable region?', in_region(P))

In [ ]:
def simulate(rhs, p, x0, T, n):
    t = np.linspace(0.0, T, n)
    sol = solve_ivp(rhs, (0.0, T), x0, t_eval=t, args=(p,), rtol=1e-9, atol=1e-11)
    return t, sol.y.T   # times (n,), states (n, m)

def add_noise(y, ub_frac, scale, rng):
    yn = y + rng.uniform(-ub_frac*scale, ub_frac*scale, size=y.shape)
    return np.clip(yn, 0.0, None)   # concentrations stay non-negative

## 2. Synthetic data

True parameters are chosen in the bistable region; the ODE is integrated from several initial
conditions (opposite corners and the symmetric point) and the trajectories are taken as
measurements.

In [ ]:
T, n = 20.0, 60
x0s = [[0.5, 5.0], [5.0, 0.5], [3.0, 3.0]]
ts, xs = [], []
for x0 in x0s:
    t, y = simulate(rhs, P, x0, T, n); ts.append(t); xs.append(y)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for x0, t, y in zip(x0s, ts, xs):
    ax[0].plot(t, y[:, 0]); ax[1].plot(t, y[:, 1])
ax[0].set_title('$x_1(t)$'); ax[1].set_title('$x_2(t)$')
for a in ax: a.set_xlabel('t')
plt.tight_layout(); plt.show()
print('final states:', *[np.round(y[-1], 3) for y in xs])

## 3. Measurement noise

Bounded uniform noise, scaled by the production gap `U - L`, is added to the trajectories; the
recovery methods see only the noisy data.

In [ ]:
rng = np.random.default_rng(0)
gap = P['U'] - P['L']
ub = 0.25                      # noise up to 25% of the characteristic scale
xs_noisy = [add_noise(y, ub, gap, rng) for y in xs]

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for t, y, yn in zip(ts, xs, xs_noisy):
    ax[0].plot(t, yn[:, 0], '.', ms=4); ax[0].plot(t, y[:, 0], 'k-', lw=.7, alpha=.5)
    ax[1].plot(t, yn[:, 1], '.', ms=4); ax[1].plot(t, y[:, 1], 'k-', lw=.7, alpha=.5)
ax[0].set_title('noisy $x_1$'); ax[1].set_title('noisy $x_2$')
plt.tight_layout(); plt.show()

## 4. Least-squares baseline (gradient matching)

Slopes `xdot` are estimated by finite differences and `(L, U, theta)` chosen so the vector
field `f(x)` matches them at the measured points (`d` fixed). No ODE solve enters the loop.

In [ ]:
def ls_recover(ts, xs, d=10.0, g=1.0):
    X = np.vstack(xs)
    DX = np.vstack([np.gradient(y, t, axis=0) for t, y in zip(ts, xs)])  # slopes from data
    def resid(z):
        L, U, th = z
        f1 = -g*X[:, 0] + hill_rep(X[:, 1], L, U, th, d)
        f2 = -g*X[:, 1] + hill_rep(X[:, 0], L, U, th, d)
        return np.concatenate([DX[:, 0] - f1, DX[:, 1] - f2])
    s = least_squares(resid, x0=[0.5, 3.0, 2.0], bounds=([0, 0, 0], [20, 20, 20]))
    return dict(L=s.x[0], U=s.x[1], th=s.x[2], d=d, g=g)

p_ls = ls_recover(ts, xs_noisy)
print('least-squares estimate (L, U, theta):', np.round([p_ls['L'], p_ls['U'], p_ls['th']], 3))
print('does it land in the bistable region?  ', in_region(p_ls))

In [ ]:
def build_tensors(ts, xs, x0s):
    # flatten all trajectories into (N,1) time, (N,m) repeated x0, (N,m) measured states
    t_d = np.concatenate([t[:, None] for t in ts])
    x_d = np.vstack(xs)
    x0_d = np.vstack([np.tile(x0, (len(t), 1)) for x0, t in zip(x0s, ts)])
    t_ic = np.zeros((len(x0s), 1)); x_ic = np.array(x0s, dtype=float)  # t=0 anchors
    to = lambda a: torch.tensor(a)
    return (to(t_d), to(x0_d), to(x_d), to(t_ic), to(x_ic))

## 5. Physics-informed neural network

A network `u_theta(t, x0)` is trained to fit the data while its time derivative satisfies the
ODE. Every model parameter is learnable and optimized jointly with the network weights: the
production parameters `(L, U, theta)`, the steepness `d`, and the decay `gamma`. They start from
a neutral init (`L ~ 0, U ~ 1, theta ~ 1, d = 5`, with `gamma = 1`), not the data-generating
values. The loss combines data mismatch, the physics residual `||du/dt - f(u)||`, and the
initial condition.

In [ ]:
# --- the physics-informed network: surrogate u_theta(t, x0) + learnable (L,U,theta,d) ---
class PINN(nn.Module):
    def __init__(self, m, param_init, T, hidden=64, depth=4, fourier_k=0):
        super().__init__()
        self.m, self.T, self.fourier_k = m, T, fourier_k
        in_dim = (2*fourier_k if fourier_k > 0 else 1) + m
        if fourier_k > 0:
            self.register_buffer('freqs', 2*np.pi*torch.arange(1, fourier_k+1).double())
        layers, d0 = [], in_dim
        for _ in range(depth):
            layers += [nn.Linear(d0, hidden), nn.Tanh()]; d0 = hidden
        layers += [nn.Linear(d0, m)]
        self.net = nn.Sequential(*layers)
        # physical parameters via positive reparametrization  p = raw^2 + eps
        self.raw = nn.ParameterDict(
            {k: nn.Parameter(torch.tensor(float(v)**0.5)) for k, v in param_init.items()})

    def phys_params(self):
        return {k: self.raw[k]**2 + 1e-6 for k in self.raw}

    def _feat(self, t):
        tn = t / self.T   # normalize time to [0,1]; the chain rule is handled by autograd
        if self.fourier_k > 0:
            return torch.cat([torch.sin(self.freqs*tn), torch.cos(self.freqs*tn)], dim=1)
        return tn

    def forward(self, t, x0):
        return self.net(torch.cat([self._feat(t), x0], dim=1))

def fit_pinn(model, data, rhs_t, steps=4000, lr=1e-3, w=(5.0, 1.0, 1.0), log=500):
    t_d, x0_d, x_d, t_ic, x_ic = data   # all tensors
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for it in range(steps):
        opt.zero_grad()
        loss_data = ((model(t_d, x0_d) - x_d)**2).mean()           # fit the measurements
        tc = t_d.clone().requires_grad_(True)                       # physics residual:
        u = model(tc, x0_d)                                         #   du/dt should equal f(u)
        grads = [torch.autograd.grad(u[:, j].sum(), tc, create_graph=True)[0]
                 for j in range(model.m)]
        du = torch.cat(grads, dim=1)
        loss_phys = ((du - rhs_t(u, model.phys_params()))**2).mean()
        loss_ic = ((model(t_ic, x_ic) - x_ic)**2).mean()           # match initial conditions
        loss = w[0]*loss_data + w[1]*loss_phys + w[2]*loss_ic
        loss.backward(); opt.step()
        if log and it % log == 0:
            print(f'  step {it:5d}  data {loss_data.item():.2e}  phys {loss_phys.item():.2e}')
    return model

In [ ]:
def rhs_t(x, p):   # the same vector field, in torch, evaluated on network outputs
    x1, x2 = x[:, 0:1], x[:, 1:2]
    d1 = -p['g']*x1 + hill_rep_t(x2, p['L'], p['U'], p['th'], p['d'])
    d2 = -p['g']*x2 + hill_rep_t(x1, p['L'], p['U'], p['th'], p['d'])
    return torch.cat([d1, d2], dim=1)

init = dict(L=0.05, U=1.0, th=1.0, d=5.0, g=1.0)   # neutral prior, NOT the truth
torch.manual_seed(0)
model = PINN(m=2, param_init=init, T=T)
data = build_tensors(ts, xs_noisy, x0s)
model = fit_pinn(model, data, rhs_t, steps=4000, lr=1e-3)

pp = {k: float(v) for k, v in model.phys_params().items()}
print('PINN estimate (L, U, theta, d):', np.round([pp['L'], pp['U'], pp['th'], pp['d']], 3))
print('does it land in the bistable region?', in_region(pp))

## 6. Region recovery across noise

For each noise level, the least-squares fit is repeated over independent noisy datasets and the
fraction satisfying the region inequality is recorded.

In [ ]:
levels = [0.0, 0.1, 0.25, 0.5]
n_trials = 20
rate = []
for ub in levels:
    r = np.random.default_rng(100); hits = 0
    for _ in range(n_trials):
        xs_n = [add_noise(y, ub, gap, r) for y in xs]
        hits += in_region(ls_recover(ts, xs_n))
    rate.append(hits / n_trials)
plt.plot([100*l for l in levels], rate, 'o-')
plt.xlabel('noise upper bound (%)'); plt.ylabel('region-recovery rate (LS)')
plt.ylim(-0.05, 1.05); plt.title('Toggle switch: region recovery vs noise'); plt.show()
print('LS region-recovery rate by noise level:', dict(zip(levels, rate)))

## Morse graph and Conley-index recovery (DSGRN, optional)

Two criteria per recovered parameter set: exact DSGRN region equality, and label-preserving
isomorphism of the **Conley-Morse graph** (recurrent Morse sets, reachability order,
Conley-index labels). Region equality implies isomorphic Morse graphs, so the second criterion
is the weaker one.

The cells below build DSGRN (C++ toolchain required) and compare the recovered and target Morse
graphs via `par_index_from_sample` (parameters to region index), `DSGRN.Blowup.ConleyMorseGraph`
(region to Morse graph), and `DSGRN.isomorphic_morse_graphs` (`node_match` on the location-free
Conley index).

In [ ]:
# Optional: building DSGRN needs a C++ toolchain (a few minutes on Colab).
# !pip -q install networkx DSGRN
DSGRN_OK = False
try:
    import DSGRN, numpy as np
    NET_SPEC = 'x : ~y\ny : ~x\n'
    _net = DSGRN.Network(NET_SPEC)
    _pg  = DSGRN.ParameterGraph(_net)
    TARGET = 4
    _mg = {}
    def conley_morse(idx):          # region index -> annotated Conley-Morse graph (cached)
        if idx not in _mg:
            _mg[idx] = DSGRN.Blowup.ConleyMorseGraph(_pg.parameter(idx))[0]
        return _mg[idx]
    def to_matrices(p):             # learned (L, U, theta) -> DSGRN L, U, T matrices
        n = 2
        L = np.zeros((n, n)); U = np.zeros((n, n)); T = np.zeros((n, n))
        L[1,0]=L[0,1]=p['L']; U[1,0]=U[0,1]=p['U']; T[0,1]=T[1,0]=p['th']
        return L, U, T
    def region_of(p):               # learned parameters -> DSGRN region index (-1 if outside)
        L, U, T = to_matrices(p)
        return DSGRN.par_index_from_sample(_pg, L, U, T)
    def morse_recovers(idx, target=TARGET):   # same Conley-Morse graph up to label-preserving iso
        return idx >= 0 and DSGRN.isomorphic_morse_graphs(conley_morse(idx), conley_morse(target))
    print('target region', TARGET, 'Conley indices:',
          [conley_morse(TARGET).vertex_label(v)[2] for v in conley_morse(TARGET).vertices()])
    DSGRN_OK = True
except Exception as e:
    print('DSGRN unavailable - skipping this section:', repr(e)[:90])

In [ ]:
if DSGRN_OK:
    # the recovered parameters from the cells above
    p_ls = ls_recover(ts, xs_noisy)
    print('recovered region index:', region_of(p_ls),
          '| exact region match:', region_of(p_ls) == TARGET,
          '| Morse/Conley match:', morse_recovers(region_of(p_ls)))

    # noise sweep: exact-region recovery vs Morse/Conley recovery (least squares)
    levels = [0.0, 0.1, 0.25, 0.5]
    print(f'{"noise":>6} | {"exact region":>12} | {"Morse/Conley":>12} | {"region-miss, Morse-ok":>20}')
    for ub in levels:
        r = np.random.default_rng(7); reg = mor = gapc = 0
        for _ in range(15):
            xs_n = [add_noise(y, ub, gap, r) for y in xs]
            idx = region_of(ls_recover(ts, xs_n))
            er = (idx == TARGET); mr = morse_recovers(idx)
            reg += er; mor += mr; gapc += (mr and not er)
        print(f'{ub:>6} | {reg:>10}/15 | {mor:>10}/15 | {gapc:>18}/15')
    # toggle: target region 4 is the unique bistable region -> the two columns coincide